# Agent testing course: plan, review and run tests with the 110 checklist designs

Free CPU, no login, no model, no API key. Turns the open [Agentic AI Test Checklist 110](https://huggingface.co/datasets/srinivaspadmanabhuni/agentic-ai-test-checklist-110) into a test plan, then fills it with agentest verdicts on a bundled sample agent (a teaching fixture, not a benchmark).

In [ ]:
import os
import pathlib

if not pathlib.Path('examples/course/test_plan.py').exists():
    if not pathlib.Path('agentest').exists():
        !git clone -q --depth 1 https://github.com/srpadmanabhuni/agentest.git
    os.chdir('agentest')
!pip install -q -e .

## 1. Load the designs

Offline by default (rows bundled in the repo). Set `USE_HUB = True` to load the published dataset at the pinned revision instead (needs `pip install datasets`; cached after the first fetch).

In [ ]:
import sys

sys.path.insert(0, 'examples/course')
import test_plan

USE_HUB = False
rows = test_plan.load_rows(hub=USE_HUB)
print(len(rows), 'test designs')
print(rows[0]['id'], '-', rows[0]['title'])

## 2. Choose the tests for your agent

Filter by priority and by the stage your agent works at (`tool`, `memory`, `output`, `plan`, `handover`, `side_effect`).

In [ ]:
plan = test_plan.select(rows, priority='P0', stage='tool')
for r in plan[:10]:
    print(r['id'], r['severity'], '-', r['title'])
print(len(plan), 'P0 tool-stage tests')

## 3. Review one design by hand

Read the acceptance criteria and the pass and fail signals. Walk them against one real conversation of your own agent before you automate anything.

In [ ]:
r = plan[0]
for k in ('objective', 'acceptance_criteria', 'pass_signal', 'fail_signal', 'fixture_needed'):
    print(k.upper(), ':', r[k], '\n')

## 4. Fill the plan with agentest verdicts

Run the checks on the bundled buggy refill agent, then on the fixed one.

In [ ]:
!python examples/course/test_plan.py --ids T1-018,T2-001,T4-003,T7-014 --verdicts-from refill_bot_buggy --out plan-buggy.csv
!python examples/course/test_plan.py --ids T1-018,T2-001,T4-003,T7-014 --verdicts-from refill_bot --out plan-fixed.csv
import csv

for name in ('plan-buggy.csv', 'plan-fixed.csv'):
    print(name)
    for row in csv.DictReader(open(name)):
        print(' ', row['id'], row['agentest_verdict'], '-', row['agentest_detail'][:80])

## 5. Your turn

Pick a design your traces cannot exercise (`n/a`) and sketch how you would implement it in your own harness. The [guide](https://github.com/srpadmanabhuni/agentest/blob/main/docs/guides/TEST-PLANNING-WITH-THE-CHECKLIST.md) shows how. If you build one, enhance and contribute it back.